# Detección de Neumonía en Radiografías de Tórax

## Descripción del Problema

La neumonía es una infección respiratoria que afecta los alvéolos pulmonares. Es una de las principales causas de muerte infantil en el mundo, y su diagnóstico temprano mediante radiografías de tórax es crucial para iniciar el tratamiento adecuado.

En este notebook construiremos un modelo de **Deep Learning** (CNN) capaz de clasificar radiografías de tórax en dos categorías:
- **NORMAL**: Pulmones sanos
- **PNEUMONIA**: Presencia de neumonía

Dataset: [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia) — Paul Mooney, Kaggle.

---
## 1. Instalación de dependencias (solo en Kaggle)

Kaggle ya tiene numpy, matplotlib y seaborn preinstalados. Si faltara alguno, esta celda lo instala.

In [ ]:
import subprocess, sys, os
if os.path.exists('/kaggle/input'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'scikit-learn'], capture_output=True)
print('Entorno listo.')

---
## 2. Importación de librerías

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import load_img, img_to_array

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(42)

print('Librerías cargadas correctamente.')

---
## 2. Configuración de rutas

El dataset está organizado en tres carpetas: `train`, `test` y `val`. Cada una contiene subcarpetas `NORMAL/` y `PNEUMONIA/` con las imágenes JPEG correspondientes.

In [ ]:
# Ruta base del dataset — funciona en local Y en Kaggle
if os.path.exists('/kaggle/input/chest-xray-pneumonia'):
    BASE_DIR = '/kaggle/input/chest-xray-pneumonia/chest_xray'
elif os.path.exists('chest_xray/chest_xray'):
    BASE_DIR = 'chest_xray/chest_xray'
else:
    BASE_DIR = 'chest_xray/chest_xray'  # fallback, ajusta si es necesario

TRAIN_DIR = os.path.join(BASE_DIR, 'train')
TEST_DIR  = os.path.join(BASE_DIR, 'test')
VAL_DIR   = os.path.join(BASE_DIR, 'val')

IMG_HEIGHT = 224
IMG_WIDTH  = 224
BATCH_SIZE = 32

print(f'Kaggle: {os.path.exists("/kaggle/input")}')
print(f'BASE_DIR: {BASE_DIR}')
print(f'Train: {TRAIN_DIR}')
print(f'Test:  {TEST_DIR}')
print(f'Val:   {VAL_DIR}')
print(f'Imágenes redimensionadas a {IMG_HEIGHT}x{IMG_WIDTH}')
print(f'Directorio actual: {os.getcwd()}')

---
## 3. Exploración del dataset

Veamos cuántas imágenes tenemos en cada split y clase. Esto nos permite detectar desbalanceo de clases.

In [ ]:
def count_images(data_dir):
    counts = {}
    for cls in os.listdir(data_dir):
        cls_path = os.path.join(data_dir, cls)
        if os.path.isdir(cls_path):
            n = len([f for f in os.listdir(cls_path) if f.lower().endswith(('.jpeg','.jpg','.png'))])
            counts[cls] = n
    return counts

for split_name, split_path in [('Train', TRAIN_DIR), ('Test', TEST_DIR), ('Val', VAL_DIR)]:
    counts = count_images(split_path)
    total = sum(counts.values())
    print(f'\n{split_name} ({total} imágenes):')
    for cls, n in counts.items():
        print(f'  {cls}: {n} ({n/total*100:.1f}%)')

### Observaciones:
- El dataset está **desbalanceado**: hay más casos de neumonía que normales (~3:1 en train).
- El split de validación es **muy pequeño** (solo 16 imágenes). Usaremos el split de test para la evaluación final.
- Podríamos usar solo train/test y reservar parte de train como validación.

**Estrategia**: Usaremos `validation_split=0.2` dentro del generador de entrenamiento para separar automáticamente el 20% de train como validación.

---
## 4. Visualización de muestras

Veamos algunas imágenes representativas de cada clase para entender las diferencias visuales.

In [ ]:
def show_sample_images(data_dir, class_name='NORMAL', n=5):
    class_path = os.path.join(data_dir, class_name)
    files = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpeg','.jpg','.png'))][:n]
    fig, axes = plt.subplots(1, n, figsize=(16, 4))
    for i, f in enumerate(files):
        img = load_img(os.path.join(class_path, f), target_size=(IMG_HEIGHT, IMG_WIDTH), color_mode='grayscale')
        axes[i].imshow(img, cmap='gray')
        axes[i].axis('off')
        axes[i].set_title(class_name)
    plt.suptitle(f'Ejemplos de {class_name}', fontsize=14)
    plt.tight_layout()
    plt.show()

show_sample_images(TRAIN_DIR, 'NORMAL')
show_sample_images(TRAIN_DIR, 'PNEUMONIA')

---
## 5. Preprocesamiento con ImageDataGenerator

Usaremos `ImageDataGenerator` de Keras para:
1. **Escalar** los píxeles al rango [0, 1]
2. **Data Augmentation** (solo en train): rotaciones, desplazamientos, zoom, volteo horizontal. Esto ayuda a generalizar y reduce overfitting.
3. **Separar validación** automática con `validation_split=0.2`

In [ ]:
# Data augmentation para entrenamiento (ayuda a prevenir overfitting)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2  # 20% para validación
)

# Para test/validación NO aplicamos augmentation, solo escalado
test_datagen = ImageDataGenerator(rescale=1./255)

print('Generadores configurados.')

### Creación de los flujos de datos

`flow_from_directory` lee automáticamente las imágenes organizadas en subcarpetas por clase y genera lotes (batches) durante el entrenamiento.

In [ ]:
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training',
    shuffle=True,
    color_mode='grayscale'
)

val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    shuffle=False,
    color_mode='grayscale'
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False,
    color_mode='grayscale'
)

print(f'\nClases: {train_generator.class_indices}')  # {'NORMAL': 0, 'PNEUMONIA': 1}

---
## 6. Construcción del modelo CNN

Diseñaremos una **Red Neuronal Convolucional (CNN)** desde cero con la siguiente arquitectura:

| Capa | Tipo | Detalle |
|------|------|---------|
| 1 | Conv2D + ReLU | 32 filtros, 3×3 |
 | MaxPooling | 2×2 |
| 2 | Conv2D + ReLU | 64 filtros, 3×3 |
 | MaxPooling | 2×2 |
| 3 | Conv2D + ReLU | 128 filtros, 3×3 |
 | MaxPooling | 2×2 |
| 4 | Conv2D + ReLU | 256 filtros, 3×3 |
 | MaxPooling | 2×2 |
| 5 | Flatten | — |
| 6 | Dense + ReLU | 512 neuronas |
| 7 | Dropout | 50% |
| 8 | Dense + Sigmoid | 1 neurona (salida binaria) |

**Elecciones de diseño:**
- `BatchNormalization` después de cada Conv2D para estabilizar el entrenamiento.
- `Dropout(0.5)` para reducir overfitting.
- `Sigmoid` en la salida para clasificación binaria.
- `Adam` como optimizador con learning rate inicial de 1e-4.

In [ ]:
model = Sequential([
    # Bloque 1
    Conv2D(32, (3,3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, 1)),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2,2)),

    # Bloque 2
    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2,2)),

    # Bloque 3
    Conv2D(128, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2,2)),

    # Bloque 4
    Conv2D(256, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2,2)),

    # Cabecera clasificadora
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  # salida binaria
])

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

### Callbacks

Configuramos tres callbacks para mejorar el entrenamiento:
- **EarlyStopping**: detiene el entrenamiento si la validación no mejora tras 10 épocas.
- **ReduceLROnPlateau**: reduce el learning rate si la pérdida de validación se estanca.
- **ModelCheckpoint**: guarda el mejor modelo encontrado durante el entrenamiento.

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1),
    ModelCheckpoint('best_model.keras', monitor='val_loss', save_best_only=True, verbose=1)
]

print('Callbacks configurados.')

---
## 7. Entrenamiento

Calculamos los pasos por época y comenzamos el entrenamiento.

**Nota**: Dependiendo de tu hardware (CPU/GPU), esto puede tomar varios minutos.

In [ ]:
steps_per_epoch = train_generator.samples // BATCH_SIZE
validation_steps = val_generator.samples // BATCH_SIZE

print(f'Pasos por época (train): {steps_per_epoch}')
print(f'Pasos de validación:     {validation_steps}')
# Aumentar a 30-50 para convergencia real (tarda ~30 min por época en CPU)
EPOCHS = 50
print(f'Épocas máximas:          {EPOCHS}')

history = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS,
    validation_data=val_generator,
    validation_steps=validation_steps,
    callbacks=callbacks,
    verbose=1
)

---
## 8. Curvas de entrenamiento

Visualizamos la evolución de la pérdida (loss) y la precisión (accuracy) durante el entrenamiento. Esto nos permite detectar overfitting.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(history.history['loss'], label='Train Loss', linewidth=2)
ax1.plot(history.history['val_loss'], label='Val Loss', linewidth=2)
ax1.set_title('Evolución de la Pérdida', fontsize=13)
ax1.set_xlabel('Época')
ax1.set_ylabel('Loss (binary crossentropy)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
ax2.plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
ax2.set_title('Evolución de la Precisión', fontsize=13)
ax2.set_xlabel('Época')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_epoch = np.argmin(history.history['val_loss']) + 1
best_val_acc = max(history.history['val_accuracy'])
print(f'Mejor época: {best_epoch}')
print(f'Mejor accuracy en validación: {best_val_acc:.4f}')

---
## 9. Evaluación en el conjunto de test

Evaluamos el modelo con datos que nunca ha visto durante el entrenamiento. Esta es la métrica que realmente importa.

In [ ]:
test_loss, test_acc = model.evaluate(test_generator, verbose=0)
print(f'Pérdida en test:  {test_loss:.4f}')
print(f'Accuracy en test: {test_acc:.4f}')

### Predicciones detalladas

Obtenemos las predicciones del modelo sobre todo el conjunto de test para calcular métricas más informativas.

In [ ]:
# Resetear el generador para asegurar predicciones correctas
test_generator.reset()

y_pred_prob = model.predict(test_generator, verbose=1)
y_pred = (y_pred_prob > 0.5).astype(int).ravel()
y_true = test_generator.classes[:len(y_pred)]

print('\nMatriz de confusión:')
cm = confusion_matrix(y_true, y_pred)
print(cm)

print('\nReporte de clasificación:')
print(classification_report(y_true, y_pred, target_names=['NORMAL', 'PNEUMONIA']))

### Interpretación de métricas

- **Accuracy**: porcentaje total de aciertos.
- **Precision**: de los que el modelo clasificó como neumonía, ¿cuántos realmente lo eran?
- **Recall (Sensibilidad)**: de los pacientes con neumonía, ¿cuántos detectó el modelo?
- **F1-Score**: media armónica de precisión y recall.

En diagnóstico médico, **el recall (sensibilidad) es la métrica más crítica**: queremos minimizar los falsos negativos (pacientes con neumonía que el modelo clasifica como normales).

### Matriz de confusión visual

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['NORMAL', 'PNEUMONIA'], 
            yticklabels=['NORMAL', 'PNEUMONIA'])
plt.title('Matriz de Confusión — Test Set', fontsize=13)
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.show()

### Curva ROC y AUC

La curva ROC muestra la relación entre la tasa de verdaderos positivos (TPR) y la tasa de falsos positivos (FPR) para distintos umbrales. El **AUC** (Área Bajo la Curva) resume el rendimiento en un solo número: 1.0 es perfecto, 0.5 es aleatorio.

In [ ]:
fpr, tpr, thresholds = roc_curve(y_true, y_pred_prob)
auc = roc_auc_score(y_true, y_pred_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'CNN (AUC = {auc:.3f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Clasificador aleatorio (AUC = 0.5)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasa de Falsos Positivos (FPR)')
plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
plt.title('Curva ROC — Test Set', fontsize=13)
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()

---
## 10. Visualización de predicciones

Veamos cómo se comporta el modelo en imágenes individuales del conjunto de test.

In [ ]:
def predict_and_show(img_path, model, threshold=0.5):
    """Carga una imagen, la pasa por el modelo y muestra el resultado."""
    img = load_img(img_path, target_size=(IMG_HEIGHT, IMG_WIDTH), color_mode='grayscale')
    img_array = img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prob = model.predict(img_array, verbose=0)[0][0]
    pred_class = 'PNEUMONIA' if prob > threshold else 'NORMAL'
    confidence = prob if prob > threshold else 1 - prob

    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    ax.imshow(img, cmap='gray')
    ax.axis('off')
    color = 'red' if pred_class == 'PNEUMONIA' else 'green'
    ax.set_title(f'{pred_class}\nConfianza: {confidence:.2%}', 
                 fontsize=13, color=color, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Probamos con algunas imágenes de test
import random

print('Ejemplos de NORMAL (test):')
normal_files = [os.path.join(TEST_DIR, 'NORMAL', f) for f in os.listdir(os.path.join(TEST_DIR, 'NORMAL'))
                if f.lower().endswith(('.jpeg','.jpg','.png'))]
for f in random.sample(normal_files, 3):
    predict_and_show(f, model)

print('Ejemplos de PNEUMONIA (test):')
pneumonia_files = [os.path.join(TEST_DIR, 'PNEUMONIA', f) for f in os.listdir(os.path.join(TEST_DIR, 'PNEUMONIA'))
                   if f.lower().endswith(('.jpeg','.jpg','.png'))]
for f in random.sample(pneumonia_files, 3):
    predict_and_show(f, model)

---
## 11. Análisis de errores

Examinemos los casos donde el modelo se equivoca. Esto nos da pistas sobre cómo mejorar.

In [ ]:
# Identificar falsos positivos y falsos negativos
test_generator.reset()
filenames = test_generator.filenames

fp_indices = np.where((y_true == 0) & (y_pred == 1))[0]  # dijo neumonía, era normal
fn_indices = np.where((y_true == 1) & (y_pred == 0))[0]  # dijo normal, era neumonía

print(f'Falsos Positivos (FP): {len(fp_indices)} — el modelo clasificó como NEUMONÍA cuando era NORMAL')
print(f'Falsos Negativos (FN): {len(fn_indices)} — el modelo clasificó como NORMAL cuando era NEUMONÍA')

def show_error_samples(indices, title, max_show=5):
    n = min(len(indices), max_show)
    if n == 0:
        print(f'  ¡No hay {title}!')
        return
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    for i, idx in enumerate(indices[:n]):
        img_path = os.path.join(TEST_DIR, filenames[idx])
        img = load_img(img_path, target_size=(IMG_HEIGHT, IMG_WIDTH), color_mode='grayscale')
        axes[i].imshow(img, cmap='gray')
        axes[i].axis('off')
        axes[i].set_title(f'Conf: {y_pred_prob[idx][0]:.2%}', fontsize=10)
    plt.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()

if len(fp_indices) > 0:
    show_error_samples(fp_indices, 'Falsos Positivos (NORMAL predicho como NEUMONÍA)')
if len(fn_indices) > 0:
    show_error_samples(fn_indices, 'Falsos Negativos (NEUMONÍA predicho como NORMAL)')

---
## 12. Conclusiones

### Resumen de resultados

| Métrica | Valor |
|---------|-------|
| Accuracy en test | (completar tras ejecución) |
| AUC | (completar tras ejecución) |
| Sensitivity (Recall NEUMONÍA) | (completar) |
| Specificity (Recall NORMAL) | (completar) |

### Hallazgos clave

1. **¿Funciona el modelo?** Un AUC > 0.95 indicaría que el modelo es excelente para esta tarea.
2. **¿Hay overfitting?** Si la pérdida de entrenamiento sigue bajando mientras la de validación sube, hay overfitting (el EarlyStopping debería mitigarlo).
3. **¿Qué tipo de errores comete?** Los falsos negativos (neumonía no detectada) son el error más peligroso en este contexto clínico.

### Posibles mejoras
- **Transfer Learning**: usar una red pre-entrenada (ResNet, DenseNet, EfficientNet) en lugar de entrenar desde cero.
- **Balanceo de clases**: usar pesos de clase (`class_weight` en Keras) para compensar el desbalance.
- **Más datos de validación**: el split de validación actual es muy pequeño; usar cross-validation o un split más grande.
- **Ensamblado**: combinar múltiples modelos para predicciones más robustas.
- **Explicabilidad**: usar Grad-CAM para visualizar qué regiones del pulmón activan la decisión del modelo.

In [ ]:
# Guardamos el modelo final para uso futuro
model.save('pneumonia_cnn_final.keras')
print('Modelo guardado como pneumonia_cnn_final.keras')